In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Authentication credentials for the aacuser MongoDB account

username = "aacuser"
password = "aacuser1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Encode the Grazioso Salvare logo image for display in the dashboard
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),

    # Header row with logo and unique identifier
    html.Div([
        html.A(
            # Logo links to Grazioso Salvare home page as required
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '100px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.Center([
            html.B(html.H1('Grazioso Salvare Dashboard')),
            # Unique identifier for this dashboard
            html.H3('Created by Suraj Hamal')
        ])
    ], style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'}),

    html.Hr(),

    # Interactive filter options using radio buttons
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water Rescue'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain Rescue'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster Rescue'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px'}
        )
    ]),

    html.Hr(),

    # Interactive data table populated with AAC outcomes data
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        # Enable column filtering
        filter_action="native",
        # Enable sorting by clicking column headers
        sort_action="native",
        sort_mode="multi",
        # Allow selecting a single column
        column_selectable="single",
        # Allow selecting a single row for the map callback
        row_selectable="single",
        # Select the first row by default so the map has data on load
        selected_rows=[0],
        # Enable pagination with 10 rows per page
        page_action="native",
        page_current=0,
        page_size=10,
    ),

    html.Br(),
    html.Hr(),

    # Side-by-side layout for pie chart and geolocation map
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            # Pie chart showing breed distribution
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            # Geolocation map showing selected animal location
            html.Div(
                id='map-id',
                className='col s12 m6',
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # Filter data based on the selected rescue type using MongoDB queries
    # Queries use breed, sex, and age criteria from the Dashboard Specifications Document

    if filter_type == 'Water Rescue':
        # Water rescue dogs: Labrador Retriever Mix, Chesapeake Bay Retriever, Newfoundland
        # Preferred sex: Intact Female, Age: 26-156 weeks
        filtered_df = pd.DataFrame.from_records(db.read({
            'animal_type': 'Dog',
            'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }))

    elif filter_type == 'Mountain Rescue':
        # Mountain rescue dogs: German Shepherd, Alaskan Malamute, Old English Sheepdog,
        # Siberian Husky, Rottweiler
        # Preferred sex: Intact Male, Age: 26-156 weeks
        filtered_df = pd.DataFrame.from_records(db.read({
            'animal_type': 'Dog',
            'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog',
                               'Siberian Husky', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }))

    elif filter_type == 'Disaster Rescue':
        # Disaster rescue dogs: Doberman Pinscher, German Shepherd, Golden Retriever,
        # Bloodhound, Rottweiler
        # Preferred sex: Intact Male, Age: 20-300 weeks
        filtered_df = pd.DataFrame.from_records(db.read({
            'animal_type': 'Dog',
            'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever',
                               'Bloodhound', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }))

    else:
        # Reset: return all documents unfiltered
        filtered_df = pd.DataFrame.from_records(db.read({}))

    # Remove the _id column to prevent data table crash
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    return filtered_df.to_dict('records')


@app.callback(Output('graph-id', "children"),
        [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # Return empty if no data is available yet
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # Return empty if DataFrame has no data
    if dff.empty:
        return []
    
    # If there are too many unique breeds (e.g. unfiltered Reset view),
    # group all but the top 10 most common breeds into "Other"
    # to keep the chart readable and consistently sized
    breed_counts = dff['breed'].value_counts()
    
    if len(breed_counts) > 10:
        top_breeds = breed_counts.nlargest(10).index
        dff = dff.copy()
        dff['breed'] = dff['breed'].apply(lambda x: x if x in top_breeds else 'Other')

    # Display a pie chart showing the distribution of breeds in the filtered data
    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Preferred Animal Breeds'
            )
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # Return empty list if no column is selected yet on initial load
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://luckycanary-southmotor-3000.codio.io/proxy/8050/
